In [ ]:
print("jo")

In [ ]:
%pip install transformers datasets huggingface_hub accelerate


In [ ]:
%%python --version


In [ ]:
%pip install torch torchvision transformers datasets huggingface_hub


In [ ]:
import torch

# This tells PyTorch to use the GPU if it's available, otherwise fallback to CPU
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"

print(f"You are currently using a: {device.upper()}")

In [ ]:
import torchvision.models as models

# 1. Load the ResNet-18 model. "weights='DEFAULT'" loads a pre-trained version.
resnet = models.resnet18(weights='DEFAULT')
print(type(resnet))

# 2. Move the model to our device (GPU)
resnet = resnet.to(device)
resnet.eval()  # Set the model to evaluation mode (important for inference)

print("ResNet successfully loaded and moved to:", device)

In [ ]:
from huggingface_hub import notebook_login

notebook_login()


In [ ]:
from datasets import load_dataset

# We use streaming=True to avoid downloading 100+ GB of data!
print("Connecting to ImageNet-1k...")
dataset = load_dataset("imagenet-1k", split="train", streaming=True, trust_remote_code=True)
print(type(dataset))

# Look at the very first image in the dataset
iterator = iter(dataset)
first_sample = next(iterator)

print("\nDataset Labels:", first_sample['label'])
first_sample['image']


In [ ]:
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import requests

# 1. Load the model and the processor (which prepares our images/text for the model)
model_id = "openai/clip-vit-base-patch32"
### STUDENT CODE HERE ###

clip_processor = CLIPProcessor.from_pretrained(model_id)
clip_model = CLIPModel.from_pretrained(model_id)

clip_model.to(device)

# 2. Get a sample image from the internet
url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(url, stream=True).raw)

# 3. Give the model some text options to guess what is in the image
text_options = ["a photo of a cat", "a photo of a dog", "a photo of a car"]

# 4. Process the inputs and run the model!
### STUDENT CODE HERE ###

inputs = clip_processor(text=text_options, images=image, return_tensors="pt", padding=True).to(device)
new_inputs = {}

for i, j in inputs.items():
    new_inputs[i] = j.to(device)

outputs = clip_model(**inputs)

# 5. Look at the results
logits_per_image = outputs.logits_per_image
probs = logits_per_image.softmax(dim=1)

print("Probability it is a cat:", probs[0][0].item())
print("Probability it is a dog:", probs[0][1].item())
print("Probability it is a car:", probs[0][2].item())

In [ ]:
from torchvision import transforms

# ResNet requires images to be resized and normalized in a very specific way
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Process the image and move it to the GPU
### STUDENT CODE HERE ###

input_tensor = preprocess(image).unsqueeze(0).to(device)

print(input_tensor.shape)  # Should be [1, 3, 224, 224]

# Run the model
with torch.inference_mode():
    ### STUDENT CODE HERE ###
    output = resnet(input_tensor)
print(output.shape) # Should be [1, 1000]

# Get the highest predicted score
predicted_class = output.argmax(dim=1).item()
print(f"ResNet predicts this is class ID: {predicted_class}")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
llm_model = AutoModelForCausalLM.from_pretrained(model_id).to(device)

prompt = "What is the capital of Poland?"
inputs = tokenizer(prompt, return_tensors="pt").to(device)
outputs = llm_model.generate(**inputs, max_new_tokens=50)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
from transformers import Blip2Processor, Blip2ForConditionalGeneration

model_id = "Salesforce/blip2-opt-2.7b"

blip_processor = Blip2Processor.from_pretrained(model_id)
blip_model = Blip2ForConditionalGeneration.from_pretrained(model_id).to(device)

image = first_sample['image']


question = "What is the thing on the image?"
inputs = blip_processor(images=image, text=question, return_tensors="pt").to(device)

outputs = blip_model.generate(**inputs, max_new_tokens=50)
answer = blip_processor.decode(outputs[0], skip_special_tokens=True)

print("Question:", question)
print("Answer:", answer)